# Retrieval Baselines — TF-IDF vs LaBSE vs E5 vs BGE-M3 vs AfroLM

Three retrievers for the multilingual health QA challenge, all scored with the
competition's **exact** whitespace-tokenizer ROUGE so local numbers track the leaderboard.

| Retriever | Similarity | Why it might win |
|---|---|---|
| **TF-IDF char n-gram** | char_wb (3,5) cosine | The starter baseline. Script-agnostic, strong, free. |
| **LaBSE** | dense cosine | 109-language sentence embeddings; strong general baseline. |
| **multilingual-E5-base** | dense cosine | Retrieval-tuned; needs `query:`/`passage:` prefixes. |
| **BGE-M3** | dense cosine | SOTA multilingual retrieval encoder; strong on low-resource. |
| **AfroLM** | dense cosine (mean-pooled) | African-pretrained; best Akan/Luganda/Amharic *coverage*. |

The core idea: on a ROUGE metric, **returning a real human-written training answer beats generating one.**
The only question is which similarity function finds the best training answer to return.

All three share one interface: fit on (train questions → train answers), then for each
val/test question retrieve the nearest train question's answer. Per-subset indices so we
never retrieve an Akan answer for a Swahili question.

## 1 — Setup

In [1]:
# !pip install -q scikit-learn pandas numpy rouge-score sentence-transformers transformers torch
# print("done")

In [2]:
import re, numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from rouge_score import rouge_scorer

SEED = 42; np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


## 2 — Load data

In [3]:
DATA_DIR = Path(".")
train = pd.read_csv(DATA_DIR/"Train.csv")
val   = pd.read_csv(DATA_DIR/"Val.csv")
test  = pd.read_csv(DATA_DIR/"Test.csv")

QCOL, ACOL, GCOL, IDCOL = "input", "output", "subset", "ID"

for df, name in [(train,"train"),(val,"val"),(test,"test")]:
    for c in [QCOL, GCOL]:
        df[c] = df[c].fillna("").astype(str).str.strip()
    if ACOL in df.columns:
        df[ACOL] = df[ACOL].fillna("").astype(str).str.strip()
train = train[(train[QCOL]!="")&(train[ACOL]!="")].reset_index(drop=True)
val   = val[(val[QCOL]!="")&(val[ACOL]!="")].reset_index(drop=True)
print(f"train {len(train)}  val {len(val)}  test {len(test)}")
print(train[GCOL].value_counts())

train 29814  val 6686  test 2618
subset
Eng_Uga    7623
Aka_Gha    4455
Eng_Gha    4443
Eng_Eth    3915
Lug_Uga    3383
Eng_Ken    2080
Swa_Ken    2070
Amh_Eth    1845
Name: count, dtype: int64


## 3 — Competition scorer (EXACT)

Whitespace tokenizer, no lowercasing, no normalization — copied from the official
starter notebook. This is the only scorer whose numbers track the leaderboard.

In [4]:
class WhitespaceTokenizer:
    def tokenize(self, text):
        if text is None: return []
        return str(text).strip().split()

_SCORER = rouge_scorer.RougeScorer(["rouge1","rougeL"],
                                   tokenizer=WhitespaceTokenizer(), use_stemmer=False)

def compute_rouge(preds, refs):
    r1, rl = [], []
    for p, r in zip(preds, refs):
        s = _SCORER.score(str(r), str(p))
        r1.append(s["rouge1"].fmeasure); rl.append(s["rougeL"].fmeasure)
    return {"rouge1_f1": float(np.mean(r1)) if r1 else 0.0,
            "rougeL_f1": float(np.mean(rl)) if rl else 0.0}

def rouge_by_subset(preds, refs, subs):
    sub = np.array(subs); rows=[]
    for s in np.unique(sub):
        m = sub==s
        rows.append({"subset":s, "n":int(m.sum()),
                     **compute_rouge([p for p,k in zip(preds,m) if k],
                                     [x for x,k in zip(refs,m) if k])})
    df = pd.DataFrame(rows)
    df.loc[len(df)] = {"subset":"OVERALL(micro)","n":len(preds),
                       **compute_rouge(preds, refs)}
    return df

## 4 — Retriever 1: TF-IDF char n-gram (the baseline to beat)

Per-subset char_wb TF-IDF + cosine nearest neighbour. `lowercase=False` preserves
case for non-Latin scripts. Char n-grams are script-agnostic — no tokenization needed.

In [5]:
class TfidfRetriever:
    def __init__(self, ngram_range=(3,5), max_features=200_000):
        self.ngram_range=ngram_range; self.max_features=max_features; self.models={}
    def _fit_one(self, q, a):
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=self.ngram_range,
                              min_df=1, max_features=self.max_features, lowercase=False)
        X = vec.fit_transform(q)
        nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(X)
        return {"vec":vec, "nn":nn, "ans":np.array(a,dtype=object)}
    def fit(self, df, qcol, acol, gcol):
        for g, grp in df.groupby(gcol):
            self.models[g] = self._fit_one(grp[qcol].tolist(), grp[acol].tolist())
        return self
    def predict(self, df, qcol, gcol):
        out=[""]*len(df); sims=[0.0]*len(df)
        pos = {idx:i for i,idx in enumerate(df.index)}
        for g, grp in df.groupby(gcol):
            if g not in self.models:
                # fall back to any model if unseen subset
                model = next(iter(self.models.values()))
            else:
                model = self.models[g]
            X = model["vec"].transform(grp[qcol].tolist())
            dist, idx = model["nn"].kneighbors(X)
            for row_idx, d, ix in zip(grp.index, dist[:,0], idx[:,0]):
                out[pos[row_idx]] = model["ans"][ix]
                sims[pos[row_idx]] = 1.0 - d
        return out, sims

tfidf = TfidfRetriever().fit(train, QCOL, ACOL, GCOL)
tfidf_val, tfidf_sim = tfidf.predict(val, QCOL, GCOL)
print("TF-IDF char(3,5):")
print(rouge_by_subset(tfidf_val, val[ACOL].tolist(), val[GCOL].tolist()).round(4).to_string(index=False))

TF-IDF char(3,5):
        subset    n  rouge1_f1  rougeL_f1
       Aka_Gha 1114     0.2832     0.1674
       Amh_Eth  462     0.1455     0.1353
       Eng_Eth  564     0.5158     0.4975
       Eng_Gha 1104     0.2582     0.1707
       Eng_Ken  390     0.5989     0.5606
       Eng_Uga 1688     0.5164     0.4710
       Lug_Uga  846     0.5155     0.4935
       Swa_Ken  518     0.6031     0.5672
OVERALL(micro) 6686     0.4207     0.3654


## 5 — Retriever 2 & 3: Dense embedding retrieval

A single class that takes any encoder (a sentence-transformers model OR a mean-pooled
HF model like AfroLM) and does per-subset cosine nearest-neighbour over question
embeddings. Swap the encoder to compare.

In [6]:
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

def mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

class HFEncoder:
    """Mean-pooled embeddings from any HF encoder (e.g. AfroLM, XLM-R)."""
    def __init__(self, model_name, max_length=128, batch_size=64):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
        self.max_length=max_length; self.batch_size=batch_size
    @torch.no_grad()
    def encode(self, texts):
        embs=[]
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i+self.batch_size]
            enc = self.tok(batch, padding=True, truncation=True,
                           max_length=self.max_length, return_tensors="pt").to(DEVICE)
            out = self.model(**enc).last_hidden_state
            e = mean_pool(out, enc["attention_mask"])
            e = F.normalize(e, p=2, dim=1)
            embs.append(e.cpu().numpy())
        return np.vstack(embs)

class STEncoder:
    """sentence-transformers encoder (LaBSE, multilingual-E5, etc.)."""
    def __init__(self, model_name):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name, device=DEVICE)
    def encode(self, texts):
        return self.model.encode(texts, normalize_embeddings=True,
                                 show_progress_bar=False, batch_size=64)

class EmbeddingRetriever:
    def __init__(self, encoder): self.encoder=encoder; self.models={}
    def fit(self, df, qcol, acol, gcol):
        for g, grp in df.groupby(gcol):
            emb = self.encoder.encode(grp[qcol].tolist())
            nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(emb)
            self.models[g] = {"nn":nn, "ans":np.array(grp[acol].tolist(),dtype=object)}
        return self
    def predict(self, df, qcol, gcol):
        out=[""]*len(df); sims=[0.0]*len(df)
        pos = {idx:i for i,idx in enumerate(df.index)}
        for g, grp in df.groupby(gcol):
            model = self.models.get(g) or next(iter(self.models.values()))
            emb = self.encoder.encode(grp[qcol].tolist())
            dist, idx = model["nn"].kneighbors(emb)
            for row_idx, d, ix in zip(grp.index, dist[:,0], idx[:,0]):
                out[pos[row_idx]] = model["ans"][ix]
                sims[pos[row_idx]] = 1.0 - d
        return out, sims

### 5a — Multilingual sentence-transformer embeddings (LaBSE, E5, BGE-M3)

Three retrieval-capable multilingual encoders, all via sentence-transformers, all top-1.

- **LaBSE** — 109 languages, robust general-purpose sentence embeddings.
- **multilingual-E5-base** — retrieval-tuned. Requires `query:` / `passage:` prefixes
  (handled by the `E5Encoder` wrapper below); skipping the prefixes silently hurts E5.
- **BGE-M3** — current SOTA multilingual retrieval encoder, strong on low-resource languages.

All three are downloaded on first use; BGE-M3 is the largest (~2.2 GB).

In [ ]:
# E5 needs instruction prefixes; wrap STEncoder to add them.
class E5Encoder(STEncoder):
    def encode(self, texts):
        return super().encode([f"query: {t}" for t in texts])

dense_specs = {
    "E5":     lambda: E5Encoder("intfloat/multilingual-e5-base"),
    "BGE-M3": lambda: STEncoder("BAAI/bge-m3"),
}

dense_preds = {}   # name -> list of retrieved answers on val
for name, make in dense_specs.items():
    print(f"\n=== {name} ===")
    retr = EmbeddingRetriever(make()).fit(train, QCOL, ACOL, GCOL)
    preds, _ = retr.predict(val, QCOL, GCOL)
    dense_preds[name] = preds
    globals()[f"retr_{name.replace('-','').lower()}"] = retr   # keep for submission
    print(rouge_by_subset(preds, val[ACOL].tolist(), val[GCOL].tolist()).round(4).to_string(index=False))

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).



=== E5 ===


W0528 18:41:48.746000 27028 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

        subset    n  rouge1_f1  rougeL_f1
       Aka_Gha 1114     0.2835     0.1696
       Amh_Eth  462     0.1659     0.1554
       Eng_Eth  564     0.5403     0.5201
       Eng_Gha 1104     0.2837     0.1882
       Eng_Ken  390     0.7804     0.7602
       Eng_Uga 1688     0.6916     0.6581
       Lug_Uga  846     0.4361     0.4086
       Swa_Ken  518     0.7156     0.6881
OVERALL(micro) 6686     0.4819     0.4294

=== BGE-M3 ===


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

        subset    n  rouge1_f1  rougeL_f1
       Aka_Gha 1114     0.2822     0.1687
       Amh_Eth  462     0.1629     0.1528
       Eng_Eth  564     0.5476     0.5287
       Eng_Gha 1104     0.2826     0.1872
       Eng_Ken  390     0.7810     0.7607
       Eng_Uga 1688     0.7465     0.7211
       Lug_Uga  846     0.4271     0.3986
       Swa_Ken  518     0.7662     0.7453
OVERALL(micro) 6686     0.4986     0.4488


### 5b — AfroLM embeddings

AfroLM is encoder-only (XLM-R architecture), pretrained on 23 African languages
including Akan, Luganda, Amharic, Swahili. Mean-pooled token embeddings.
This is the one to watch on the low-resource subsets — its representations for
Akan/Luganda should be stronger than LaBSE's.

> Note: AfroLM was trained for representation, not retrieval. Raw mean-pooled
> embeddings may underperform a retrieval-tuned model like LaBSE on semantic
> matching even where its language coverage is better. Compare per-subset and
> let the numbers decide.

## 6 — Compare all three, overall and per-subset

In [8]:
def overall_row(name, preds):
    m = compute_rouge(preds, val[ACOL].tolist())
    return {"retriever":name, "rouge1_f1":round(m["rouge1_f1"],4), "rougeL_f1":round(m["rougeL_f1"],4)}

all_preds = {"TF-IDF char(3,5)": tfidf_val, **dense_preds}

comparison = pd.DataFrame([overall_row(n, p) for n, p in all_preds.items()]
                          ).sort_values("rouge1_f1", ascending=False)
print("OVERALL (micro), ranked:")
print(comparison.to_string(index=False))

print("\nPer-subset ROUGE-1 F1 (rows=subset, cols=retriever):")
def by_sub(preds):
    d = rouge_by_subset(preds, val[ACOL].tolist(), val[GCOL].tolist())
    d = d[d["subset"]!="OVERALL(micro)"]
    return d.set_index("subset")["rouge1_f1"]
side = pd.DataFrame({n: by_sub(p) for n, p in all_preds.items()}).round(4)
side["BEST"] = side.idxmax(axis=1)        # which retriever wins each subset
print(side.to_string())

print("\nPer-subset winner counts:")
print(side["BEST"].value_counts().to_string())

OVERALL (micro), ranked:
       retriever  rouge1_f1  rougeL_f1
          BGE-M3     0.4986     0.4488
              E5     0.4819     0.4294
TF-IDF char(3,5)     0.4207     0.3654

Per-subset ROUGE-1 F1 (rows=subset, cols=retriever):
         TF-IDF char(3,5)      E5  BGE-M3              BEST
subset                                                     
Aka_Gha            0.2832  0.2835  0.2822                E5
Amh_Eth            0.1455  0.1659  0.1629                E5
Eng_Eth            0.5158  0.5403  0.5476            BGE-M3
Eng_Gha            0.2582  0.2837  0.2826                E5
Eng_Ken            0.5989  0.7804  0.7810            BGE-M3
Eng_Uga            0.5164  0.6916  0.7465            BGE-M3
Lug_Uga            0.5155  0.4361  0.4271  TF-IDF char(3,5)
Swa_Ken            0.6031  0.7156  0.7662            BGE-M3

Per-subset winner counts:
BEST
BGE-M3              4
E5                  3
TF-IDF char(3,5)    1


## 7 — Build the submission from the best retriever

Pick the winner from section 6 (or do per-subset: use whichever retriever wins each
subset — often AfroLM on Akan/Luganda, LaBSE or TF-IDF elsewhere). Submission format
matches the starter: all three target columns hold the same retrieved answer.

In [10]:
# ---- Option A: single best retriever overall ----
# Handles available: tfidf, retr_labse, retr_e5, retr_bgem3, afrolm
RETRIEVERS = {
    "tfidf":  tfidf,
    "e5":     retr_e5,
    "bgem3":  retr_bgem3,
}
BEST = "bgem3"   # set to the section-6 overall winner

retriever = RETRIEVERS[BEST]
test_pred, _ = retriever.predict(test, QCOL, GCOL)

# ---- Option B (usually better): per-subset selection ----
# Uncomment to route each subset to whichever retriever won it on validation.
# Read the winners off the `side["BEST"]` table from section 6 and fill this map.
#
# SUBSET_BEST = {
#     "Aka_Gha": "afrolm", "Lug_Uga": "afrolm", "Amh_Eth": "bgem3",
#     "Swa_Ken": "bgem3",  "Eng_Uga": "labse",  "Eng_Gha": "tfidf",
#     "Eng_Ken": "labse",  "Eng_Eth": "tfidf",
# }
# test_pred = [""] * len(test)
# pos = {idx:i for i,idx in enumerate(test.index)}
# for subset, grp in test.groupby(GCOL):
#     r = RETRIEVERS[SUBSET_BEST.get(subset, BEST)]
#     preds, _ = r.predict(grp, QCOL, GCOL)
#     for row_idx, p in zip(grp.index, preds):
#         test_pred[pos[row_idx]] = p

clean = [re.sub(r"<extra_id_\d+>", "", str(p)).strip() for p in test_pred]
sub = pd.DataFrame({"ID":test[IDCOL], "TargetRLF1":clean, "TargetR1F1":clean, "TargetLLM":clean})
assert list(sub.columns)==["ID","TargetRLF1","TargetR1F1","TargetLLM"]
assert len(sub)==len(test)
sub.to_csv(f"submission_{BEST}_retrieval.csv", index=False, encoding="utf-8")
print(f"Saved submission_{BEST}_retrieval.csv  ({len(sub)} rows)")

Saved submission_bgem3_retrieval.csv  (2618 rows)


In [ ]:
# ── Build the test submission (BGE-M3 only) ───────────────────────────────────
import gc, time

# ---- Clear GPU first ----
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU cleared: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")

# ---- Build train+val corpus for test-time retrieval ----
corpus = pd.concat([train, val], ignore_index=True)
print(f"Corpus: {len(corpus)} rows (train {len(train)} + val {len(val)})")

# ---- Fit BGE-M3 on the combined corpus ----
t0 = time.time()
print("Fitting BGE-M3 on train+val ...", flush=True)
bgem3_full = EmbeddingRetriever(retr_bgem3.encoder).fit(corpus, QCOL, ACOL, GCOL)
print(f"  done in {time.time()-t0:.1f}s", flush=True)

# ---- Retrieve for test ----
t0 = time.time()
print(f"Retrieving for {len(test)} test rows ...", flush=True)
test_pred, test_sims = bgem3_full.predict(test, QCOL, GCOL)
print(f"  done in {time.time()-t0:.1f}s", flush=True)

# ---- Format + write submission ----
clean = [re.sub(r"<extra_id_\d+>", "", str(p)).strip() for p in test_pred]
sub = pd.DataFrame({"ID": test[IDCOL], "TargetRLF1": clean, "TargetR1F1": clean, "TargetLLM": clean})
assert list(sub.columns) == ["ID", "TargetRLF1", "TargetR1F1", "TargetLLM"]
assert len(sub) == len(test)
sub.to_csv("submission_bgem3_retrieval.csv", index=False, encoding="utf-8")
print(f"Saved submission_bgem3_retrieval.csv ({len(sub)} rows)")

GPU cleared: 1.78 GB free / 6.44 GB total
Corpus: 36500 rows (train 29814 + val 6686)
Fitting BGE-M3 on train+val ...
  done in 183.7s
Retrieving for 2618 test rows ...
  done in 17.3s
Saved submission_bgem3_retrieval.csv (2618 rows)


: 

## 8 — Where to go from here

Retrieval is the floor, not the ceiling. Once you know which similarity wins per subset:

1. **Per-subset retriever selection** — use AfroLM where it wins (likely Akan/Luganda),
   LaBSE/TF-IDF elsewhere. Cheap, and usually beats any single retriever overall.
2. **Retrieve top-k, not top-1** — pull the 3-5 nearest answers and pick/merge. Helps
   when the single nearest neighbour is a near-miss.
3. **Rerank** — retrieve top-k with a fast model, rerank with a cross-encoder. The
   standard retrieval-quality lever.
4. **RAG** — retrieve top-k training answers, feed them to your fine-tuned mT0 as
   context, let it synthesize. Combines your generative work with retrieval's grounding.
   This is likely where beating the pack (and charmq) lives — retrieval gets everyone
   to ~0.6, so the edge is in what you add on top.
5. **Better embeddings** — try multilingual-E5-large, BGE-M3 (strong multilingual
   retrieval model), or fine-tune an encoder on (question, answer) pairs with contrastive
   loss so the embedding space is tuned for THIS retrieval task.